In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b",max_tokens="1000")


### summarization middleware

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### messagebased summarization
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("messages",10),
            keep=("messages",4),
        )
    ]
)

In [5]:
# run with thread_id 
config = {"configurable":{"thread_id":"tt-1"}}

In [6]:
#test data 

In [ ]:
questions = [
    "what is ai?, give answer in one line ",
    "what is ml? , give answer in one line ",
    "what is no 1 country? , give answer in one line",
    "what is richest country? ,give answer in one line",
    "what is happiest country? ,give answer in one line",
    "who is best ufc fighter? , give answer in one line" 
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")



### token size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star , $350/night , spa , pool ,gym
    2.City Inn - 4 star, $180/night, business center
    3.Budget Stay - 3 star, $75/night, free wifi"""
    
    
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

#token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star , $350/night , spa , pool ,gym
    2.City Inn - 4 star, $180/night, business center
    3.Budget Stay - 3 star, $75/night, free wifi"""
    
    
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("fraction",0.005),
            keep=("fraction",0.002),
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

#token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [ ]:
# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # groq:qwen/qwen3.8-27b context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])


### Human in the Loop MiddleWare

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [28]:
agent=create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

### approve

In [23]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
 )

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='683d2327-89de-4bf7-9faa-b55ee449abf2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5xmpbty4e', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.171911382, 'completion_tokens_details': None, 'prompt_time': 0.026493011, 'prompt_tokens_details': None, 'queue_time': 0.049849208, 'total_time': 0.198404393}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17333-116f-771c-957b-a6a7b1d87c80-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [25]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Done! The email has been sent to john@test.com with the subject "Hello" and body "How are you?".


In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='683d2327-89de-4bf7-9faa-b55ee449abf2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5xmpbty4e', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.171911382, 'completion_tokens_details': None, 'prompt_time': 0.026493011, 'prompt_tokens_details': None, 'queue_time': 0.049849208, 'total_time': 0.198404393}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17333-116f-771c-957b-a6a7b1d87c80-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

### reject

In [30]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
 )

In [31]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a8307f11-7f61-433f-8f2f-6ae701b6a7d3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '51707jvz7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.178596884, 'completion_tokens_details': None, 'prompt_time': 0.026547126, 'prompt_tokens_details': None, 'queue_time': 0.050556722, 'total_time': 0.20514401}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1733f-375a-7318-88b6-b1b7874a888e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'H

In [32]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: I attempted to send the email to john@test.com with the subject "Hello" and body "How are you?", but the action was rejected and not executed.

Let me know if you'd like to make any changes to the email (e.g., different recipient, subject, or body) or if you'd like me to try again.


In [33]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a8307f11-7f61-433f-8f2f-6ae701b6a7d3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '51707jvz7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.178596884, 'completion_tokens_details': None, 'prompt_time': 0.026547126, 'prompt_tokens_details': None, 'queue_time': 0.050556722, 'total_time': 0.20514401}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1733f-375a-7318-88b6-b1b7874a888e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'H

### editing

In [34]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)


In [35]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='a6a3dea0-475a-43b7-acd8-daa066c5194f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h68ka3mbx', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 374, 'total_tokens': 428, 'completion_time': 0.15749289, 'completion_tokens_details': None, 'prompt_time': 0.029943423, 'prompt_tokens_details': None, 'queue_time': 0.109995032, 'total_time': 0.187436313}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17343-642b-76b4-9af7-1c0e4c010e4d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'reci

In [36]:

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email was sent successfully. Note that a human reviewer modified the email before it was sent — it ultimately went to **correct@email.com** with the subject **"Corrected Subject"** and body **"This was edited by human before sending"**.

Let me know if you'd like to send a follow-up or make any changes.


In [37]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='a6a3dea0-475a-43b7-acd8-daa066c5194f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h68ka3mbx', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 374, 'total_tokens': 428, 'completion_time': 0.15749289, 'completion_tokens_details': None, 'prompt_time': 0.029943423, 'prompt_tokens_details': None, 'queue_time': 0.109995032, 'total_time': 0.187436313}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17343-642b-76b4-9af7-1c0e4c010e4d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'reci